# Who: Jacopo

**Task:** Make text-classifier on the consult texts with agenda codes as labels

*Ingredients*: 
* probably best to use a trusted CNN but I know that you want to use a transformer so..just follow the recipe you can probably find online ;)
* Your text column:  ```text```, contains a concatenation of ```SpecialismeNaam, title, section_title_display_original, section_text```
* The target variable: use the target value you extracted from the afspraken set
* for the training use cross-validation: we are interested firstly in the performance on the test-sets, AND we are interested in **inspecting** the samples in the test-set that are high confidence false negative (potentially patients that should be included) and high confidence false negatives. 


In [44]:
import pandas as pd
from tokenizers import ByteLevelBPETokenizer
from transformers import PreTrainedTokenizerFast
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import os

In [3]:
df_consult_text = pd.read_parquet(r"T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240826\consult_texts.parquet")

In [4]:
os.chdir(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240805')
df_afsprak = pd.read_sas('afspraken_20240805.sas7bdat', encoding='latin1')
df_afsprak.studyId_0831 = df_afsprak.studyId_0831.astype('uint64')
df_afsprak = df_afsprak.drop(['AfspCode', 'Agenda', 'period_start',\
       'period_end', 'Spec_Code', 'Specialisme', 'AgendaNaam', 'AgendaComment',\
       'AgendaKostenplaats', 'Description', 'AfspraakNummer', 'SubAgenda',\
       'serviceProv_Org_value', 'AfspraakStatus', 'status_code_original',\
       'status_display_original', 'type1_code', 'type1_display',\
       'type1_system', 'type2_code', 'type2_display', 'type2_system',\
       'type3_code', 'type3_display', 'type3_system', 'index_date'],axis='columns')
df_afsprak = df_afsprak.groupby('studyId_0831', sort=True).agg(lambda x: x.to_list())


In [5]:
def isEligible(x:pd.Series):
    if 'NFALG' in x or 'NFGEH' in x or 'NFVMI+' in x or 'NFVAL+' in x:
        return 1
    else:
        return 0

In [ ]:
df_afsprak['label'] = df_afsprak.index_AfspCode.apply(isEligible)
df_afsprak = df_afsprak[['label']]
df_afsprak

In [7]:
df_consult = pd.read_sas(
    r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240822\consult_20240822.sas7bdat',
        encoding='latin1')
df_consult.studyId_0831 = df_consult.studyId_0831.astype('uint64')

In [ ]:
df_consult = df_consult.drop(['Consult_id', 'ConsultStatus', 'type_code_original',
                              'type_display_original','created', 'title', 'author_usercode',
                              'initial_author_usercode', 'custodian_Organization_value',
                              'SpecialismeNaam', 'initial_custodian_Organization_v',
                              'SpecialismeNaamInitial', 'LocationCode', 'LocationName',
                              'PatientProtected', 'section_title_code_original',
                              'section_title_display_original', 'section_text',
                              'section_text_contentType', 'orderedBy', 'index_date'],axis='columns')
df_consult = df_consult.groupby('studyId_0831').last()
df_consult

In [ ]:
df_merged = pd.merge(df_consult_text, df_consult, right_index=True,left_on='studyId_0831',how='left')
df_merged

In [10]:
df_merged['delta'] = (df_merged.date_x - df_merged.date_y).dt.days
df_merged_filt = df_merged.query('delta >= -365 and delta <= 0')

In [11]:
indices = []
def take_longer_consult(x):
    all_consults = x.to_list()
    max_len = -1
    for i,c in enumerate(all_consults):
        if len(c) > max_len:
            tmp = c
            idx = i
            max_len = len(c)
    
    indices.append(x.index[idx])
    return tmp


In [12]:
df_merged_f_g = df_merged_filt[['studyId_0831','text']]\
                    .groupby('studyId_0831')\
                        .agg(take_longer_consult)

In [ ]:
df_merged_filt['delta'].loc[indices].hist(bins=100)

In [14]:
df_merged_f_g['delta'] = df_merged_filt['delta'].loc[indices]

In [113]:
#df_cooked_labels = pd.merge(df_cooked, df_afsprak, right_index=True,left_on='studyId_0831',how='left')
df_consult_text = pd.merge(df_consult_text, df_afsprak, right_index=True,left_on='studyId_0831',how='left')

---

Be careful uncommenting here, dataset generation code chunk...

In [22]:
#df_cooked_labels
#generate_dataset(df_cooked_labels, path=r'C:\Users\jvitale\text_dataset')

---

Be careful uncommenting here, tokenizer training, necessary if change random seed

In [66]:
#os.chdir(r'C:\Users\jvitale')
#
#rng = np.random.default_rng(seed=42)
#test_split  = 0.2
#train_split = 1 - test_split
#
#train_df = df_consult_text.groupby('label').sample(frac=train_split,random_state=rng) # nice and stratified
#
#paths = [str(x) for x in Path("whole_text_dataset/train/").glob("**/*.txt")]
#
## Initialize a tokenizer
#tokenizer = ByteLevelBPETokenizer()
#
## Customize training
#tokenizer.train(files=paths, vocab_size=5000, min_frequency=2, special_tokens=[
#    "<s>",
#    "<pad>",
#    "</s>",
#    "<unk>",
#    "<mask>",
#])
#os.chdir(r'C:\Users\jvitale\OneDrive - UMC Utrecht\ODIN\sandboxes\07_consults\pretrained')
#tokenizer.save_model('DutchEHRTokenizer')

['DutchEHRTokenizer\\vocab.json', 'DutchEHRTokenizer\\merges.txt']

---

Here below exploratory code to tune steps for the model:

Steps will be integrated into the `forward()` method of the model

ergo, code below is unnecessary but kept just to waste space

In [115]:
os.chdir(r'C:\Users\jvitale\OneDrive - UMC Utrecht\ODIN\sandboxes\07_consults')
tokenizer = ByteLevelBPETokenizer(vocab=r'pretrained\DutchEHRTokenizer\vocab.json',
                                  merges=r'pretrained\DutchEHRTokenizer\merges.txt',
                                  add_prefix_space = True,
                                  )

In [116]:
df_cooked = pd.DataFrame(columns=['ids'])

def to_ids(text):
    return tokenizer.encode(text)

def to_padded_encoding(encoded_text):
    if len(encoded_text.ids) == 5000:
        return encoded_text
    elif len(encoded_text.ids) > 5000:
        encoded_text.truncate(5000)
    else:
        encoded_text.pad(5000)
        
    return encoded_text

In [119]:
df_cooked['encoding']        = df_merged_f_g.text.apply(to_ids)
df_cooked['padded_encoding'] = df_cooked.encoding.apply(to_padded_encoding)
df_cooked['ids']             = df_cooked.padded_encoding.apply(lambda x: x.ids)
df_cooked['num_token']       = df_cooked.padded_encoding.apply(lambda x: len(x))


In [123]:
df_cooked = df_cooked.drop('encoding',axis='columns')
df_cooked

,ids,padded_encoding,num_token
studyId_0831,,,
7992019,"[404, 527, 570, 572, 30, 408, 203, 588, 30, 72...","Encoding(num_tokens=5000, attributes=[ids, typ...",5000
9812359,"[404, 527, 570, 572, 30, 408, 203, 588, 30, 72...","Encoding(num_tokens=5000, attributes=[ids, typ...",5000
22291801,"[404, 527, 570, 572, 30, 408, 203, 588, 30, 72...","Encoding(num_tokens=5000, attributes=[ids, typ...",5000
29822615,"[404, 527, 570, 572, 30, 408, 203, 588, 30, 72...","Encoding(num_tokens=5000, attributes=[ids, typ...",5000
42120023,"[404, 527, 570, 572, 30, 408, 203, 588, 30, 72...","Encoding(num_tokens=5000, attributes=[ids, typ...",5000
...,...,...,...
67374258279,"[404, 527, 570, 572, 30, 408, 203, 588, 30, 72...","Encoding(num_tokens=5000, attributes=[ids, typ...",5000
67384735347,"[404, 527, 570, 572, 30, 408, 203, 588, 30, 72...","Encoding(num_tokens=5000, attributes=[ids, typ...",5000
67387668117,"[404, 527, 570, 572, 30, 408, 203, 588, 30, 72...","Encoding(num_tokens=5000, attributes=[ids, typ...",5000


---